# Binary Image-Text Matching Dataset Preparation

## Introduction

This notebook prepares the Flickr30k dataset for a supervised binary image-text matching task.

The goal is to transform the original image-caption structure into labeled pairs:

- `1` — the caption correctly describes the image
- `0` — the caption belongs to a different image and is treated as a negative example

This prepared dataset will later be used to train and evaluate baseline and neural binary classification models.

The main objectives of this notebook are:

- Load the Flickr30k dataset.
- Use the predefined train, validation, and test splits.
- Generate positive image-caption pairs.
- Generate negative image-caption pairs.
- Create a balanced supervised dataset.
- Save or structure the prepared data for later modeling notebooks.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [2]:
import random
import pandas as pd

from datasets import load_dataset

from src.config import DATASET_NAME, RANDOM_SEED

## Loading the Dataset

The dataset is loaded from Hugging Face Datasets. The original split information is stored in the `split` column, so it will be used to separate the data into train, validation, and test subsets.

In [3]:
dataset = load_dataset(DATASET_NAME)
data = dataset["test"]

data

Dataset({
    features: ['image', 'caption', 'sentids', 'split', 'img_id', 'filename'],
    num_rows: 31014
})

## Dataset Preparation Strategy

The original Flickr30k dataset contains one image paired with five human-written captions. While this structure is well suited for image captioning and retrieval tasks, it must be transformed into a supervised dataset for binary image-text matching.

Positive samples are created by pairing each image with its corresponding captions. Negative samples are generated by pairing each image with captions randomly selected from different images within the same dataset split. This ensures that the model learns to distinguish between matching and non-matching image-caption pairs while preventing information leakage between the training, validation, and test sets.

To maintain a balanced binary classification dataset, one negative sample is generated for each positive sample, resulting in an equal number of positive and negative examples.


## Train / Validation / Test Split

The Flickr30k dataset is loaded as a single Hugging Face split, but the original dataset split is stored in the `split` column.

To avoid data leakage, train, validation, and test subsets are separated before generating positive and negative image-caption pairs. Negative captions will only be sampled from the same split as the image.

In [4]:
train_data = data.filter(lambda sample: sample["split"] == "train")
val_data = data.filter(lambda sample: sample["split"] == "val")
test_data = data.filter(lambda sample: sample["split"] == "test")

len(train_data), len(val_data), len(test_data)

Filter:   0%|          | 0/31014 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31014 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31014 [00:00<?, ? examples/s]

(29000, 1014, 1000)

### Observation

The dataset contains 29,000 training images, 1,014 validation images, and 1,000 test images. These predefined splits will be used throughout the project to keep model training, validation, and final evaluation properly separated.

## Positive Pair Generation

Positive pairs are created by pairing each image with its original human-written captions.

Since each image in Flickr30k has five captions, each image produces five positive image-caption pairs. These pairs receive the label `1`, meaning that the caption correctly describes the image.

In [7]:
from src.data.preparation import (
    create_positive_pairs,
)

train_positive_pairs = create_positive_pairs(train_data)
val_positive_pairs = create_positive_pairs(val_data)
test_positive_pairs = create_positive_pairs(test_data)

train_positive_pairs.shape, val_positive_pairs.shape, test_positive_pairs.shape

((145000, 4), (5070, 4), (5000, 4))

In [14]:
train_positive_pairs.head(10)

,img_id,filename,caption,label
0,0,1000092795.jpg,Two young guys with shaggy hair look at their hands while hanging out in the yard.,1
1,0,1000092795.jpg,"Two young, White males are outside near many bushes.",1
2,0,1000092795.jpg,Two men in green shirts are standing in a yard.,1
3,0,1000092795.jpg,A man in a blue shirt standing in a garden.,1
4,0,1000092795.jpg,Two friends enjoy time spent together.,1
5,1,10002456.jpg,Several men in hard hats are operating a giant pulley system.,1
6,1,10002456.jpg,Workers look down from up above on a piece of equipment.,1
7,1,10002456.jpg,Two men working on a machine wearing hard hats.,1
8,1,10002456.jpg,Four men on top of a tall structure.,1
9,1,10002456.jpg,Three men on a large rig.,1


### Observation

Positive pair generation produces five labeled examples for each image, matching the original Flickr30k structure. The training split contains 145,000 positive image-caption pairs, while the validation and test splits contain 5,070 and 5,000 positive pairs respectively.

These examples form the positive class for the binary image-text matching task.

## Negative Pair Generation

Negative pairs are created by pairing each image with captions that belong to different images from the same dataset split.

To avoid data leakage, negative captions are sampled only from within the current split. For each image, the number of generated negative pairs matches the number of positive captions, resulting in a balanced binary classification dataset.

In [9]:
from src.data.preparation import create_negative_pairs

train_negative_pairs = create_negative_pairs(train_data, seed=RANDOM_SEED)
val_negative_pairs = create_negative_pairs(val_data, seed=RANDOM_SEED)
test_negative_pairs = create_negative_pairs(test_data, seed=RANDOM_SEED)

train_negative_pairs.shape, val_negative_pairs.shape, test_negative_pairs.shape

((145000, 4), (5070, 4), (5000, 4))

In [13]:
train_negative_pairs.head(10)

,img_id,filename,caption,label
0,0,1000092795.jpg,A man in a green shirt positions a sheep for clipping with electric sheers.,0
1,0,1000092795.jpg,Four young boys are running down a sidewalk.,0
2,0,1000092795.jpg,A man with his shirt off sweats profusely as he takes a break by a wall.,0
3,0,1000092795.jpg,An equestrian rider in uniform jumps their white horse over a fence.,0
4,0,1000092795.jpg,A guy sitting on his truck reading a post on his social network.,0
5,1,10002456.jpg,A man is making a phone call.,0
6,1,10002456.jpg,A black and white dog running on the beach while a man stands behind it.,0
7,1,10002456.jpg,Native woman in blue dress prepares pineapple and other fruits.,0
8,1,10002456.jpg,An old man is examining his camera while a younger man is singing nearby.,0
9,1,10002456.jpg,Men play a game on the street.,0


### Observation

Negative pair generation produces the same number of examples as positive pair generation. Each negative pair contains an image and a randomly selected caption from a different image within the same split.

This creates a balanced supervised dataset for binary image-text matching while preventing captions from validation or test images from being used during training.

## Binary Dataset Creation

Positive and negative pairs are combined into a single supervised dataset for binary image-text matching.

The resulting dataset is balanced, containing an equal number of matching (`label = 1`) and non-matching (`label = 0`) image-caption pairs. After combining the pairs, the rows are shuffled to avoid any ordering bias during model training.

In [15]:
from src.data.preparation import create_binary_dataset

train_pairs = create_binary_dataset(
    train_positive_pairs,
    train_negative_pairs,
    seed=RANDOM_SEED,
)

val_pairs = create_binary_dataset(
    val_positive_pairs,
    val_negative_pairs,
    seed=RANDOM_SEED,
)

test_pairs = create_binary_dataset(
    test_positive_pairs,
    test_negative_pairs,
    seed=RANDOM_SEED,
)

train_pairs.shape, val_pairs.shape, test_pairs.shape

((290000, 4), (10140, 4), (10000, 4))

In [16]:
train_pairs["label"].value_counts()

label
0    145000
1    145000
Name: count, dtype: int64

## Prepared Dataset Summary

In [17]:
summary = pd.DataFrame(
    {
        "Split": ["Train", "Validation", "Test"],
        "Images": [
            len(train_data),
            len(val_data),
            len(test_data),
        ],
        "Positive Pairs": [
            len(train_positive_pairs),
            len(val_positive_pairs),
            len(test_positive_pairs),
        ],
        "Negative Pairs": [
            len(train_negative_pairs),
            len(val_negative_pairs),
            len(test_negative_pairs),
        ],
        "Total Pairs": [
            len(train_pairs),
            len(val_pairs),
            len(test_pairs),
        ],
    }
)

summary

,Split,Images,Positive Pairs,Negative Pairs,Total Pairs
0,Train,29000,145000,145000,290000
1,Validation,1014,5070,5070,10140
2,Test,1000,5000,5000,10000


## Notebook Summary

In this notebook, the Flickr30k dataset was transformed into a supervised binary classification dataset for image-text matching.

Positive pairs were created by pairing each image with its original captions, while negative pairs were generated by randomly sampling captions from different images within the same dataset split. The resulting datasets are balanced and maintain the original train, validation, and test separation, preventing data leakage.

These prepared datasets provide the foundation for training and evaluating the baseline and neural image-text matching models developed in [the following notebooks](https://github.com/Maxstef/flickr30k-multimodal-retrieval/blob/main/notebooks/03_baseline_models.ipynb).
